# Scenario 5 — Cross-Corpus Domain Generalisation
## Encoder Notebook (RoBERTa-base) — FIXED

**Fixes applied:**
1. Email text normalisation (URLs, emails, whitespace)
2. Class-weighted loss to fix precision collapse
3. Decision threshold tuning on validation set
4. Few-shot target-domain mixing (100 samples from test corpus added to training)

**Cross-tests:**
- Train on CEAS-08 → Test on TREC-07
- Train on Enron → Test on Ling-Spam

**Dataset:** `puyang2025/seven-phishing-email-datasets`

**Model:** RoBERTa-base

In [ ]:
!nvidia-smi
!pip install -q "numpy==1.26.4" "scipy==1.12.0"
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers accelerate bitsandbytes peft datasets scikit-learn pandas tqdm

In [2]:
import os, re, time, warnings
import numpy as np, pandas as pd, torch
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss
from datasets import load_dataset
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

# All three encoder models — used in Cell 5 loop
ENCODER_MODELS = {
    "BERT":       "bert-base-uncased",
    "RoBERTa":    "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
}

# ── FIX 1: Email text normalisation ─────────────────────────────────────────
def clean_email(text):
    """Normalise surface-level corpus-specific patterns."""
    text = str(text)
    text = re.sub(r'http\S+|www\.\S+', '<URL>', text)
    text = re.sub(r'\S+@\S+\.\S+', '<EMAIL>', text)
    text = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '<IP>', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text.lower()

def evaluate(y_true, y_pred, name="", ms=None):
    r = {"Model":     name,
         "Accuracy":  f"{accuracy_score(y_true, y_pred):.4f}",
         "Precision": f"{precision_score(y_true, y_pred, zero_division=0):.4f}",
         "Recall":    f"{recall_score(y_true, y_pred, zero_division=0):.4f}",
         "F1":        f"{f1_score(y_true, y_pred, average='binary', zero_division=0):.4f}"}
    if ms: r["ms/sample"] = f"{ms:.2f}"
    return r

class EmailDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.enc = tokenizer(list(texts), padding="max_length", truncation=True,
                             max_length=max_len, return_tensors="pt")
        self.labels = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return {k: v[i] for k, v in self.enc.items()}, self.labels[i]

Device: cuda
GPU: Tesla T4


In [3]:
# Dataset: puyang2025/seven-phishing-email-datasets — 203k emails across 7 corpora
print("Loading puyang2025/seven-phishing-email-datasets...")
ds_p = load_dataset("puyang2025/seven-phishing-email-datasets", split="train")
df_puy = ds_p.to_pandas()

# Auto-detect text columns
text_candidates = [c for c in df_puy.columns
                   if c.lower() in ("subject", "body", "text", "email_text", "message", "content")]
print(f"Detected text columns: {text_candidates}")

if len(text_candidates) == 1:
    df_puy["text"] = df_puy[text_candidates[0]].fillna("").str.strip()
else:
    df_puy["text"] = (df_puy[text_candidates]
                      .fillna("")
                      .apply(lambda row: " ".join(str(v) for v in row if str(v).strip()), axis=1)
                      .str.strip())

# FIX 1: Apply normalisation
print("Normalising email text...")
df_puy["text"] = df_puy["text"].apply(clean_email)

df_puy = (df_puy[["text", "label", "dataset_name"]]
          .drop_duplicates("text")
          .dropna()
          .reset_index(drop=True))
df_puy["label"] = df_puy["label"].astype(int)
print(f"Total: {len(df_puy):,}")
print(df_puy["dataset_name"].value_counts().to_string())

Loading puyang2025/seven-phishing-email-datasets...


README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/184M [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/22.7M [00:00<?, ?B/s]

eval.parquet:   0%|          | 0.00/22.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/162413 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/40604 [00:00<?, ? examples/s]

Detected text columns: ['text', 'subject']
Normalising email text...
Total: 156,806
dataset_name
TREC-05     43840
TREC-07     42061
CEAS-08     27484
Enron       23626
TREC-06     12937
Assassin     4567
Ling         2291


In [4]:
# ── FIX 2: Class-weighted loss + FIX 3: threshold tuning ────────────────────
def find_best_threshold(model, val_loader, device):
    """Find the decision threshold that maximises F1 on a validation set."""
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for be, lbl in val_loader:
            logits = model(**{k: v.to(device) for k, v in be.items()}).logits
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(lbl.numpy())
    best_thresh, best_f1 = 0.5, 0.0
    for thresh in np.arange(0.3, 0.8, 0.05):
        preds = (np.array(all_probs) >= thresh).astype(int)
        f1 = f1_score(all_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    print(f"  Best threshold: {best_thresh:.2f} (val F1: {best_f1:.4f})")
    return best_thresh

def finetune_encoder(model_id, name, X_tr, y_tr, X_val, y_val, X_te, y_te,
                     epochs=3, lr=2e-5, batch=16, max_len=256):
    tok   = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2).to(DEVICE)

    tr_dl  = DataLoader(EmailDataset(X_tr,  y_tr,  tok, max_len), batch_size=batch,   shuffle=True)
    val_dl = DataLoader(EmailDataset(X_val, y_val, tok, max_len), batch_size=batch*2)

    opt   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, len(tr_dl)//5, len(tr_dl)*epochs)

    # FIX 2: Compute class weights from training labels to fix precision collapse
    counts   = np.bincount(y_tr)
    total    = counts.sum()
    weights  = torch.tensor([total / (2 * c) for c in counts], dtype=torch.float).to(DEVICE)
    loss_fn  = CrossEntropyLoss(weight=weights)
    print(f"  Class weights: {weights.cpu().numpy().round(3)}")

    for ep in range(1, epochs+1):
        model.train(); total_loss = 0
        for be, lbl in tqdm(tr_dl, desc=f"{name} ep{ep}"):
            be  = {k: v.to(DEVICE) for k, v in be.items()}
            out = model(**be)
            loss = loss_fn(out.logits, lbl.to(DEVICE))   # weighted loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); opt.zero_grad()
            total_loss += loss.item()
        print(f"  Epoch {ep} loss: {total_loss/len(tr_dl):.4f}")

    # FIX 3: Tune decision threshold on validation set
    best_thresh = find_best_threshold(model, val_dl, DEVICE)

    # Evaluate on test set with tuned threshold
    te_dl = DataLoader(EmailDataset(X_te, y_te, tok, max_len), batch_size=batch*2)
    model.eval(); all_probs = []; t0 = time.time()
    with torch.no_grad():
        for be, _ in te_dl:
            logits = model(**{k: v.to(DEVICE) for k, v in be.items()}).logits
            all_probs.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    ms    = (time.time() - t0) / len(y_te) * 1000
    preds = (np.array(all_probs) >= best_thresh).astype(int)

    del model; torch.cuda.empty_cache()
    result = evaluate(y_te, preds, name, ms)
    result["Threshold"] = f"{best_thresh:.2f}"
    return result

In [5]:
# ── FIX 4: Few-shot target-domain mixing (100 samples from test corpus) ──────
FEW_SHOT_N = 100   # number of target-domain samples to add to training

# Cross-test A: CEAS-08 → TREC-07
df_c = df_puy[df_puy.dataset_name=="CEAS-08"].sample(min(4000, len(df_puy[df_puy.dataset_name=="CEAS-08"])), random_state=42)
df_t = df_puy[df_puy.dataset_name=="TREC-07"].sample(min(1000, len(df_puy[df_puy.dataset_name=="TREC-07"])), random_state=42)

# Reserve FEW_SHOT_N from test set for training mix — rest is evaluation
df_t_few  = df_t.sample(FEW_SHOT_N, random_state=0)
df_t_eval = df_t.drop(df_t_few.index)

X_tr_A  = np.concatenate([df_c["text"].values,     df_t_few["text"].values])
y_tr_A  = np.concatenate([df_c["label"].values,    df_t_few["label"].values])
X_val_A = df_c["text"].values[:200]
y_val_A = df_c["label"].values[:200]
X_te_A  = df_t_eval["text"].values
y_te_A  = df_t_eval["label"].values

print(f"Train (CEAS-08 + {FEW_SHOT_N} TREC-07): {len(X_tr_A):,} | Test (TREC-07): {len(X_te_A):,}")
row_A = finetune_encoder("roberta-base", "RoBERTa (CEAS-08 → TREC-07)",
                          X_tr_A, y_tr_A, X_val_A, y_val_A, X_te_A, y_te_A)

# Cross-test B: Enron → Ling-Spam
df_e = df_puy[df_puy.dataset_name=="Enron"].sample(min(3000, len(df_puy[df_puy.dataset_name=="Enron"])), random_state=42)
df_l = df_puy[df_puy.dataset_name=="Ling"].sample(min(800,  len(df_puy[df_puy.dataset_name=="Ling"])),  random_state=42)

df_l_few  = df_l.sample(FEW_SHOT_N, random_state=0)
df_l_eval = df_l.drop(df_l_few.index)

X_tr_B  = np.concatenate([df_e["text"].values,     df_l_few["text"].values])
y_tr_B  = np.concatenate([df_e["label"].values,    df_l_few["label"].values])
X_val_B = df_e["text"].values[:200]
y_val_B = df_e["label"].values[:200]
X_te_B  = df_l_eval["text"].values
y_te_B  = df_l_eval["label"].values

print(f"Train (Enron + {FEW_SHOT_N} Ling): {len(X_tr_B):,} | Test (Ling-Spam): {len(X_te_B):,}")
row_B = finetune_encoder("roberta-base", "RoBERTa (Enron → Ling-Spam)",
                          X_tr_B, y_tr_B, X_val_B, y_val_B, X_te_B, y_te_B)

print("\n" + "="*60)
print("SCENARIO 5 — ENCODER (CROSS-CORPUS) RESULTS — FIXED")
print("="*60)
print(pd.DataFrame([row_A, row_B]).to_string(index=False))
print("\nFixes: class-weighted loss + threshold tuning + few-shot mixing + text normalisation")

Train (CEAS-08 + 100 TREC-07): 4,100 | Test (TREC-07): 900


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

  Epoch 1 loss: 0.2367


RoBERTa (Enron → Ling-Spam) ep2:   0%|          | 0/194 [00:00<?, ?it/s]

  Epoch 2 loss: 0.0455


RoBERTa (Enron → Ling-Spam) ep3:   0%|          | 0/194 [00:00<?, ?it/s]

  Epoch 3 loss: 0.0139
  Best threshold: 0.30 (val F1: 0.9936)

SCENARIO 5 — ENCODER (CROSS-CORPUS) RESULTS — FIXED
                      Model Accuracy Precision Recall     F1 ms/sample Threshold
RoBERTa (CEAS-08 → TREC-07)   0.9100    0.9548 0.8661 0.9083     16.15      0.30
RoBERTa (Enron → Ling-Spam)   0.9843    0.9470 0.9690 0.9579     16.37      0.30

Fixes: class-weighted loss + threshold tuning + few-shot mixing + text normalisation
